# Kvasir-SEG: Polyp-PVT vs CFP + Polyp-PVT

This notebook compares the official Polyp-PVT architecture against the same model preceded by `mtlearn.layers.ConnectedFilterPreprocessingLayer` (CFP). Both variants start from the same PVTv2-B2 ImageNet checkpoint (`pvt_v2_b2.pth`) and use the same Kvasir-SEG split, training schedule, loss, and metrics.

Default execution mode is `smoke`: it downloads/prepares resources and runs the pipeline on a tiny subset. Set `run_mode = "full"` in the configuration cell to run the complete experiment. Device selection is `cuda -> mps -> cpu`; on MPS the notebook uses `384x384` by default because Polyp-PVT's SAM block calls `adaptive_avg_pool2d`, and PyTorch MPS currently fails for the original `352x352` feature size.

Sources:

- Kvasir-SEG: https://datasets.simula.no/kvasir-seg/
- Polyp-PVT: https://github.com/DengPingFan/Polyp-PVT
- PVTv2-B2 checkpoint mirror used by default: https://huggingface.co/Anonymity/pvt_pretrained/blob/main/pvt_v2_b2.pth

In [1]:
from __future__ import annotations

import csv
import json
import math
import os
import random
import shutil
import ssl
import subprocess
import sys
import time
import urllib.request
import zipfile
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'mtlearn' / 'python' / 'mtlearn').is_dir():
            return candidate
    raise RuntimeError('Could not locate mtlearn repository root.')


REPO_ROOT = find_repo_root()
# Prefer the repository Python package. Do not force build-jupyter bindings here: that
# build may be older than the Python layer and miss newer binding signatures.
python_src = REPO_ROOT / 'mtlearn' / 'python'
if str(python_src) not in sys.path:
    sys.path.insert(0, str(python_src))

print('repo:', REPO_ROOT)
print('python:', sys.executable)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('mps available:', bool(hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()))


repo: /Users/wonderalexandre/GitHub/mtlearn
python: /opt/anaconda3/bin/python
torch: 2.10.0
cuda available: False
mps available: True


## Configuration

Use `RUN_MODE = "smoke"` for a small, fast run that exercises the data, cache, model, loss, and metrics path. Use `RUN_MODE = "full"` for the real experiment.

In [2]:
@dataclass
class ExperimentConfig:
    seed: int = 42
    run_mode: str = 'smoke'  # 'smoke' or 'full'
    device_preference: str = 'auto'  # 'auto', 'cuda', 'mps', or 'cpu'
    image_size: int = 352
    batch_size: int = 4
    mps_image_size: int = 384
    mps_max_batch_size: int = 2
    num_workers: int = 0
    full_epochs: int = 100
    smoke_epochs: int = 1
    smoke_train_samples: int = 4
    smoke_val_samples: int = 2
    smoke_test_samples: int = 2
    val_fraction: float = 0.10
    test_fraction: float = 0.12
    lr_backbone: float = 1e-4
    lr_cfp: float = 1e-3
    weight_decay_backbone: float = 1e-4
    weight_decay_cfp: float = 1e-7
    grad_clip: float = 0.5
    cfp_identity_p0: float = 0.995
    cfp_beta_f: float = 1.0
    cfp_clamp: float = 12.0


CFG = ExperimentConfig()


def select_device(preference: str = 'auto') -> torch.device:
    preference = preference.lower()
    if preference not in {'auto', 'cuda', 'mps', 'cpu'}:
        raise ValueError(f'Unsupported device preference: {preference}')
    if preference in {'auto', 'cuda'} and torch.cuda.is_available():
        return torch.device('cuda')
    if preference in {'auto', 'mps'} and hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    if preference == 'cuda':
        raise RuntimeError('CUDA was requested but is not available.')
    if preference == 'mps':
        raise RuntimeError('MPS was requested but is not available.')
    return torch.device('cpu')


DEVICE = select_device(CFG.device_preference)
if DEVICE.type == 'mps':
    # Polyp-PVT's SAM block uses AdaptiveAvgPool2d(output_size=6). With the
    # original 352 input, the relevant feature map is 44x44, and PyTorch MPS
    # does not implement non-divisible adaptive pooling. 384 -> 48x48 works.
    if CFG.image_size != CFG.mps_image_size:
        print(f'MPS selected: changing image_size from {CFG.image_size} to {CFG.mps_image_size}.')
        CFG.image_size = CFG.mps_image_size
    if CFG.batch_size > CFG.mps_max_batch_size:
        print(f'MPS selected: reducing batch_size from {CFG.batch_size} to {CFG.mps_max_batch_size}.')
        CFG.batch_size = CFG.mps_max_batch_size

DATA_ROOT = REPO_ROOT / 'dat' / 'kvasir_seg'
EXTERNAL_ROOT = REPO_ROOT / 'external_models'
POLYP_REPO = EXTERNAL_ROOT / 'Polyp-PVT'
PVT_PRETRAINED_PATH = POLYP_REPO / 'pretrained_pth' / 'pvt_v2_b2.pth'
OUT_DIR = REPO_ROOT / 'notebooks' / 'experiments' / 'out_kvasir_polyp_pvt_cfp'
SPLIT_PATH = OUT_DIR / f'splits_seed_{CFG.seed}.json'

Kvasir_URL = 'https://datasets.simula.no/downloads/kvasir-seg.zip'
PVT_B2_URL = 'https://huggingface.co/Anonymity/pvt_pretrained/resolve/main/pvt_v2_b2.pth'
POLYP_PVT_GIT = 'https://github.com/DengPingFan/Polyp-PVT.git'

OUT_DIR.mkdir(parents=True, exist_ok=True)
EXTERNAL_ROOT.mkdir(parents=True, exist_ok=True)
print(asdict(CFG))
print('device:', DEVICE)
print('output:', OUT_DIR)


MPS selected: changing image_size from 352 to 384.
MPS selected: reducing batch_size from 4 to 2.
{'seed': 42, 'run_mode': 'smoke', 'device_preference': 'auto', 'image_size': 384, 'batch_size': 2, 'mps_image_size': 384, 'mps_max_batch_size': 2, 'num_workers': 0, 'full_epochs': 100, 'smoke_epochs': 1, 'smoke_train_samples': 4, 'smoke_val_samples': 2, 'smoke_test_samples': 2, 'val_fraction': 0.1, 'test_fraction': 0.12, 'lr_backbone': 0.0001, 'lr_cfp': 0.001, 'weight_decay_backbone': 0.0001, 'weight_decay_cfp': 1e-07, 'grad_clip': 0.5, 'cfp_identity_p0': 0.995, 'cfp_beta_f': 1.0, 'cfp_clamp': 12.0}
device: mps
output: /Users/wonderalexandre/GitHub/mtlearn/notebooks/experiments/out_kvasir_polyp_pvt_cfp


In [3]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


seed_everything(CFG.seed)


## Download and Prepare Resources

The Kvasir-SEG download is direct. The PVTv2-B2 checkpoint is mirrored on Hugging Face for reproducible scripted downloads; if that fails, place `pvt_v2_b2.pth` manually at `external_models/Polyp-PVT/pretrained_pth/pvt_v2_b2.pth`.

In [4]:
def download_file(url: str, destination: Path, *, insecure_ssl: bool = False) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 0:
        print(f'already exists: {destination} ({destination.stat().st_size / 1e6:.1f} MB)')
        return

    context = ssl._create_unverified_context() if insecure_ssl else None
    request = urllib.request.Request(url, headers={'User-Agent': 'mtlearn-kvasir-polyp-pvt-cfp/1.0'})
    tmp = destination.with_suffix(destination.suffix + '.part')
    with urllib.request.urlopen(request, context=context, timeout=120) as response, tmp.open('wb') as output:
        total = response.headers.get('Content-Length')
        total = int(total) if total else None
        done = 0
        last_report = -1
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            output.write(chunk)
            done += len(chunk)
            if total:
                pct = int(100 * done / total)
                if pct // 10 != last_report // 10:
                    print(f'  {pct:3d}% ({done / 1e6:.1f} / {total / 1e6:.1f} MB)')
                    last_report = pct
    tmp.replace(destination)
    print(f'downloaded: {destination} ({destination.stat().st_size / 1e6:.1f} MB)')


def run(cmd: list[str], cwd: Path | None = None) -> None:
    print('+', ' '.join(map(str, cmd)))
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)


In [5]:
def prepare_kvasir_seg(data_root: Path = DATA_ROOT) -> tuple[Path, Path]:
    image_dir = data_root / 'images'
    mask_dir = data_root / 'masks'
    if image_dir.is_dir() and mask_dir.is_dir() and len(list(image_dir.iterdir())) >= 1000:
        print('Kvasir-SEG already prepared:', data_root)
        return image_dir, mask_dir

    archive = data_root.parent / 'kvasir-seg.zip'
    download_file(Kvasir_URL, archive, insecure_ssl=True)
    extract_dir = data_root.parent / '_kvasir_seg_extract'
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True)
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(extract_dir)

    candidates = []
    for candidate in [extract_dir, *extract_dir.rglob('*')]:
        if candidate.is_dir() and (candidate / 'images').is_dir() and (candidate / 'masks').is_dir():
            candidates.append(candidate)
    if not candidates:
        raise RuntimeError(f'Could not find images/masks directories under {extract_dir}')

    source = candidates[0]
    if data_root.exists():
        shutil.rmtree(data_root)
    data_root.mkdir(parents=True, exist_ok=True)
    shutil.move(str(source / 'images'), str(image_dir))
    shutil.move(str(source / 'masks'), str(mask_dir))
    if (source / 'kavsir_bboxes.json').exists():
        shutil.move(str(source / 'kavsir_bboxes.json'), str(data_root / 'kavsir_bboxes.json'))
    shutil.rmtree(extract_dir, ignore_errors=True)
    print('Kvasir-SEG prepared:', data_root)
    return image_dir, mask_dir


def prepare_polyp_pvt_repo(repo_dir: Path = POLYP_REPO) -> None:
    if not (repo_dir / 'lib' / 'pvt.py').is_file():
        repo_dir.parent.mkdir(parents=True, exist_ok=True)
        run(['git', 'clone', '--depth', '1', POLYP_PVT_GIT, str(repo_dir)])
    else:
        print('Polyp-PVT already available:', repo_dir)

    pvt_file = repo_dir / 'lib' / 'pvt.py'
    text = pvt_file.read_text()
    text = text.replace("path = './pretrained_pth/pvt_v2_b2.pth'", f"path = r'{PVT_PRETRAINED_PATH}'")
    text = text.replace('save_model = torch.load(path)', "save_model = torch.load(path, map_location='cpu')")
    pvt_file.write_text(text)


def prepare_pvt_pretrained(path: Path = PVT_PRETRAINED_PATH) -> None:
    if path.exists() and path.stat().st_size > 0:
        print('PVTv2-B2 checkpoint already available:', path)
        return
    download_file(PVT_B2_URL, path)


image_dir, mask_dir = prepare_kvasir_seg()
prepare_polyp_pvt_repo()
prepare_pvt_pretrained()


Kvasir-SEG already prepared: /Users/wonderalexandre/GitHub/mtlearn/dat/kvasir_seg
Polyp-PVT already available: /Users/wonderalexandre/GitHub/mtlearn/external_models/Polyp-PVT
PVTv2-B2 checkpoint already available: /Users/wonderalexandre/GitHub/mtlearn/external_models/Polyp-PVT/pretrained_pth/pvt_v2_b2.pth


## Splits and Datasets

The first run creates a deterministic split and saves it as JSON. Later runs reuse the same split. The dataset returns raw RGB tensors in `[0, 1]`; ImageNet normalization is applied inside each model wrapper so CFP operates on image intensities rather than ImageNet-normalized values.

In [6]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}


def pair_kvasir_files(image_dir: Path, mask_dir: Path) -> list[dict[str, str]]:
    images = {p.stem: p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS}
    masks = {p.stem: p for p in mask_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS}
    keys = sorted(set(images) & set(masks))
    if not keys:
        raise RuntimeError('No image/mask pairs found.')
    missing_masks = sorted(set(images) - set(masks))[:5]
    missing_images = sorted(set(masks) - set(images))[:5]
    if missing_masks or missing_images:
        print('warning: unmatched files:', {'missing_masks': missing_masks, 'missing_images': missing_images})
    return [{'name': k, 'image': str(images[k]), 'mask': str(masks[k])} for k in keys]


def create_or_load_split(pairs: list[dict[str, str]], split_path: Path = SPLIT_PATH) -> dict[str, list[dict[str, str]]]:
    if split_path.exists():
        with split_path.open() as f:
            split = json.load(f)
        print('loaded split:', split_path)
        return split

    rng = random.Random(CFG.seed)
    shuffled = pairs[:]
    rng.shuffle(shuffled)
    n = len(shuffled)
    n_test = max(1, round(n * CFG.test_fraction))
    remaining = n - n_test
    n_val = max(1, round(remaining * CFG.val_fraction))
    split = {
        'train': shuffled[: remaining - n_val],
        'val': shuffled[remaining - n_val: remaining],
        'test': shuffled[remaining:],
    }
    split_path.parent.mkdir(parents=True, exist_ok=True)
    with split_path.open('w') as f:
        json.dump(split, f, indent=2)
    print('saved split:', split_path)
    return split


pairs = pair_kvasir_files(image_dir, mask_dir)
split = create_or_load_split(pairs)
print({k: len(v) for k, v in split.items()})


loaded split: /Users/wonderalexandre/GitHub/mtlearn/notebooks/experiments/out_kvasir_polyp_pvt_cfp/splits_seed_42.json
{'train': 792, 'val': 88, 'test': 120}


In [7]:
class KvasirSegDataset(Dataset):
    def __init__(self, items: list[dict[str, str]], image_size: int = 352):
        self.items = items
        self.image_size = int(image_size)

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int):
        item = self.items[idx]
        image = Image.open(item['image']).convert('RGB')
        mask = Image.open(item['mask']).convert('L')
        image = TF.resize(image, [self.image_size, self.image_size], interpolation=InterpolationMode.BILINEAR)
        mask = TF.resize(mask, [self.image_size, self.image_size], interpolation=InterpolationMode.NEAREST)
        x = TF.to_tensor(image).float()
        y = (TF.to_tensor(mask).float() > 0.5).float()
        return x, y, item['name']


class IndexedDataset(Dataset):
    def __init__(self, base: Dataset):
        self.base = base

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx: int):
        sample = self.base[idx]
        return (sample[0], idx), sample[1]


def maybe_subset(dataset: Dataset, limit: int | None) -> Dataset:
    if limit is None or limit >= len(dataset):
        return dataset
    return Subset(dataset, list(range(limit)))


def make_loader(dataset: Dataset, *, shuffle: bool, indexed: bool = False, seed_offset: int = 0) -> DataLoader:
    wrapped = IndexedDataset(dataset) if indexed else dataset
    generator = torch.Generator()
    generator.manual_seed(CFG.seed + seed_offset)
    return DataLoader(
        wrapped,
        batch_size=CFG.batch_size,
        shuffle=shuffle,
        num_workers=CFG.num_workers,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
        generator=generator,
    )


train_dataset = KvasirSegDataset(split['train'], CFG.image_size)
val_dataset = KvasirSegDataset(split['val'], CFG.image_size)
test_dataset = KvasirSegDataset(split['test'], CFG.image_size)

if CFG.run_mode == 'smoke':
    train_dataset = maybe_subset(train_dataset, CFG.smoke_train_samples)
    val_dataset = maybe_subset(val_dataset, CFG.smoke_val_samples)
    test_dataset = maybe_subset(test_dataset, CFG.smoke_test_samples)

train_loader_base = make_loader(train_dataset, shuffle=True, indexed=False, seed_offset=1)
train_loader_cfp_prepass = make_loader(train_dataset, shuffle=False, indexed=False, seed_offset=2)
train_loader_cfp = make_loader(train_dataset, shuffle=True, indexed=True, seed_offset=1)
val_loader = make_loader(val_dataset, shuffle=False, indexed=False, seed_offset=3)
test_loader = make_loader(test_dataset, shuffle=False, indexed=False, seed_offset=4)
print('effective sizes:', len(train_dataset), len(val_dataset), len(test_dataset))


effective sizes: 4 2 2


## Model Wrappers

`PolypPVT` expects ImageNet-normalized RGB. The baseline normalizes raw images directly. The CFP model first applies CFP to raw `[0, 1]` intensities, maps the CFP output back to `[0, 1]`, and then applies ImageNet normalization.

In [8]:
if str(POLYP_REPO) not in sys.path:
    sys.path.insert(0, str(POLYP_REPO))
from lib.pvt import PolypPVT

from mtlearn import morphology
from mtlearn.layers import ConnectedFilterPreprocessingLayer


def install_compute_attributes_dtype_compat() -> None:
    """Handle native mtlearn builds that do not yet expose the dtype argument."""
    try:
        probe = np.zeros((2, 2), dtype=np.uint8)
        tree = morphology.create_max_tree(probe)
        morphology.compute_attributes(tree, [morphology.AttributeType.AREA], dtype=None)
        return
    except TypeError:
        pass

    def compute_attributes_compat(tree, attributes, output_space=morphology.NodeIdSpace.MORPHOLOGICAL_TREE, dtype=None):
        return morphology._backend.Attribute.computeAttributes(tree, attributes, output_space)

    morphology.compute_attributes = compute_attributes_compat
    print('Installed mtlearn compute_attributes compatibility shim for native bindings without dtype support.')


install_compute_attributes_dtype_compat()


class ImageNetNormalize(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return (x - self.mean.to(dtype=x.dtype, device=x.device)) / self.std.to(dtype=x.dtype, device=x.device)


class BaselinePolypPVT(nn.Module):
    def __init__(self):
        super().__init__()
        self.normalize = ImageNetNormalize()
        self.polyp_pvt = PolypPVT()

    def forward(self, x_raw: torch.Tensor):
        x_raw = x_raw.to(DEVICE, non_blocking=True)
        return self.polyp_pvt(self.normalize(x_raw))


class CFPPolypPVT(nn.Module):
    def __init__(self, cfp_layer: ConnectedFilterPreprocessingLayer):
        super().__init__()
        self.cfp = cfp_layer
        self.normalize = ImageNetNormalize()
        self.polyp_pvt = PolypPVT()

    def forward(self, x_raw_or_cached):
        x_cfp = self.cfp(x_raw_or_cached)
        x_unit = torch.clamp(x_cfp / 255.0, 0.0, 1.0)
        return self.polyp_pvt(self.normalize(x_unit))


def cfp_shape_attributes():
    shape_group = getattr(morphology.AttributeGroup, 'SHAPE', None)
    if shape_group is not None:
        return (shape_group,)
    return (
        morphology.AttributeType.AREA,
        morphology.AttributeType.COMPACTNESS,
        morphology.AttributeType.CIRCULARITY,
        morphology.AttributeType.RECTANGULARITY,
        morphology.AttributeType.GRAY_HEIGHT,
    )


def build_cfp_layer() -> ConnectedFilterPreprocessingLayer:
    layer = ConnectedFilterPreprocessingLayer(
        in_channels=3,
        filter_specs=[
            {
                'name': 'tos_shape',
                'tree_type': morphology.TreeType.TREE_OF_SHAPES,
                'attributes': cfp_shape_attributes(),
                'tos_interpolation': 'self-dual',
            }
        ],
        device=DEVICE,
        scale_mode='hybrid',
        beta_f=CFG.cfp_beta_f,
        clamp=CFG.cfp_clamp,
    )
    layer.init_identity_with_bias(p0=CFG.cfp_identity_p0)
    assert layer.out_channels == 3, f'CFP must preserve RGB channels; got {layer.out_channels}'
    return layer


Installed mtlearn compute_attributes compatibility shim for native bindings without dtype support.


/opt/anaconda3/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/opt/anaconda3/lib/python3.12/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/Users/wonderalexandre/GitHub/mtlearn/external_models/Polyp-PVT/lib/pvtv2.py:387: UserWarning: Overwriting pvt_v2_b0 in registry with lib.pvtv2.pvt_v2_b0. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/Users/wonderalexandre/GitHub/mtlearn/external_models/Polyp-PVT/lib/pvtv2.py:397: UserWarning: Overwriting pvt_v2_b1 in registry with lib.pvtv2.pvt_v2_b1. This is

## CFP Cache Prepass

The cache is built on the training dataset only. After `build_dataloader_cached`, dataset-level CFP statistics are frozen and cached normalizations are refreshed. Validation and test use the frozen training statistics.

In [9]:
cfp_layer = build_cfp_layer()
print('Building CFP cache and training statistics...')
_ = cfp_layer.build_dataloader_cached(train_loader_cfp_prepass)
cfp_layer.save_stats(str(OUT_DIR / 'cfp_train_stats.pt'))
print('CFP stats frozen:', cfp_layer._stats_frozen)
print('cached training samples:', len(cfp_layer._tree_info))


Building CFP cache and training statistics...


CFP stats frozen: True
cached training samples: 12


In [10]:
@torch.no_grad()
def cfp_identity_report(cfp_layer: ConnectedFilterPreprocessingLayer, loader: DataLoader, max_batches: int = 1) -> dict[str, float]:
    diffs = []
    for batch_i, batch in enumerate(loader):
        if batch_i >= max_batches:
            break
        images = batch[0]
        y = torch.clamp(cfp_layer(images) / 255.0, 0.0, 1.0).cpu()
        diffs.append((y - images).abs().mean().item())
    return {'mean_abs_identity_error': float(np.mean(diffs)) if diffs else float('nan')}


identity_report = cfp_identity_report(cfp_layer, train_loader_base)
print(identity_report)
with (OUT_DIR / 'cfp_identity_report.json').open('w') as f:
    json.dump(identity_report, f, indent=2)


{'mean_abs_identity_error': 0.0018162644701078534}


## Loss, Metrics, Training, and Evaluation

The loss follows Polyp-PVT's weighted BCE + weighted IoU structure. Metrics are computed from `sigmoid(P1 + P2)` without per-image min/max normalization, preserving probability calibration.

In [11]:
def structure_loss(pred: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    weit = 1 + 5 * torch.abs(F.avg_pool2d(mask, kernel_size=31, stride=1, padding=15) - mask)
    wbce = F.binary_cross_entropy_with_logits(pred, mask, reduction='none')
    wbce = (weit * wbce).sum(dim=(2, 3)) / weit.sum(dim=(2, 3)).clamp_min(1e-8)

    pred_prob = torch.sigmoid(pred)
    inter = ((pred_prob * mask) * weit).sum(dim=(2, 3))
    union = ((pred_prob + mask) * weit).sum(dim=(2, 3))
    wiou = 1 - (inter + 1) / (union - inter + 1)
    return (wbce + wiou).mean()


def unpack_batch(batch):
    if isinstance(batch, (tuple, list)) and len(batch) == 2 and isinstance(batch[0], (tuple, list)):
        model_input = batch[0]
        mask = batch[1]
        names = None
    else:
        model_input = batch[0]
        mask = batch[1]
        names = batch[2] if len(batch) > 2 else None
    return model_input, mask.to(DEVICE, non_blocking=True), names


def combined_logits(outputs, target_shape: tuple[int, int]) -> torch.Tensor:
    p1, p2 = outputs
    logits = p1 + p2
    if logits.shape[-2:] != target_shape:
        logits = F.interpolate(logits, size=target_shape, mode='bilinear', align_corners=False)
    return logits


def batch_metrics(logits: torch.Tensor, mask: torch.Tensor, threshold: float = 0.5) -> dict[str, float]:
    prob = torch.sigmoid(logits)
    pred = (prob >= threshold).float()
    dims = (1, 2, 3)
    eps = 1e-7
    soft_inter = (prob * mask).sum(dim=dims)
    soft_union = prob.sum(dim=dims) + mask.sum(dim=dims)
    soft_dice = (2 * soft_inter + eps) / (soft_union + eps)
    soft_iou = (soft_inter + eps) / (prob.sum(dim=dims) + mask.sum(dim=dims) - soft_inter + eps)

    inter = (pred * mask).sum(dim=dims)
    pred_sum = pred.sum(dim=dims)
    mask_sum = mask.sum(dim=dims)
    dice = (2 * inter + eps) / (pred_sum + mask_sum + eps)
    iou = (inter + eps) / (pred_sum + mask_sum - inter + eps)
    precision = (inter + eps) / (pred_sum + eps)
    recall = (inter + eps) / (mask_sum + eps)
    mae = torch.abs(prob - mask).mean(dim=dims)
    return {
        'soft_dice': soft_dice.mean().item(),
        'soft_iou': soft_iou.mean().item(),
        'dice': dice.mean().item(),
        'iou': iou.mean().item(),
        'precision': precision.mean().item(),
        'recall': recall.mean().item(),
        'mae': mae.mean().item(),
    }


def average_dicts(rows: list[dict[str, float]]) -> dict[str, float]:
    if not rows:
        return {}
    return {key: float(np.mean([row[key] for row in rows])) for key in rows[0]}


In [12]:
def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer) -> float:
    model.train()
    losses = []
    for batch in loader:
        model_input, mask, _ = unpack_batch(batch)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(model_input)
        logits_shape = mask.shape[-2:]
        p1, p2 = outputs
        if p1.shape[-2:] != logits_shape:
            p1 = F.interpolate(p1, size=logits_shape, mode='bilinear', align_corners=False)
            p2 = F.interpolate(p2, size=logits_shape, mode='bilinear', align_corners=False)
        loss = structure_loss(p1, mask) + structure_loss(p2, mask)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses)) if losses else float('nan')


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> dict[str, float]:
    model.eval()
    rows = []
    for batch in loader:
        model_input, mask, _ = unpack_batch(batch)
        outputs = model(model_input)
        logits = combined_logits(outputs, mask.shape[-2:])
        rows.append(batch_metrics(logits, mask))
    return average_dicts(rows)


def make_baseline_optimizer(model: BaselinePolypPVT):
    return torch.optim.AdamW(model.parameters(), lr=CFG.lr_backbone, weight_decay=CFG.weight_decay_backbone)


def make_cfp_optimizer(model: CFPPolypPVT):
    return torch.optim.AdamW(
        [
            {'params': model.cfp.parameters(), 'lr': CFG.lr_cfp, 'weight_decay': CFG.weight_decay_cfp},
            {'params': model.polyp_pvt.parameters(), 'lr': CFG.lr_backbone, 'weight_decay': CFG.weight_decay_backbone},
        ]
    )


In [13]:
def save_history(history: list[dict[str, Any]], path: Path) -> None:
    if not history:
        return
    with path.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
        writer.writeheader()
        writer.writerows(history)


def train_model(name: str, model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, optimizer: torch.optim.Optimizer, epochs: int) -> list[dict[str, Any]]:
    history = []
    best_score = -math.inf
    best_path = OUT_DIR / f'{name}_best.pt'
    for epoch in range(1, epochs + 1):
        start = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer)
        val_metrics = evaluate(model, val_loader)
        score = val_metrics.get('soft_dice', float('-inf'))
        row = {
            'epoch': epoch,
            'model': name,
            'train_loss': train_loss,
            **{f'val_{k}': v for k, v in val_metrics.items()},
            'seconds': time.time() - start,
        }
        history.append(row)
        print(row)
        if score > best_score:
            best_score = score
            payload = {'model': model.state_dict(), 'epoch': epoch, 'val_metrics': val_metrics, 'config': asdict(CFG)}
            torch.save(payload, best_path)
    save_history(history, OUT_DIR / f'{name}_history.csv')
    return history


## Execute Experiment

In `smoke` mode this runs one epoch on a tiny subset. In `full` mode it runs the full schedule.

In [14]:
epochs = CFG.smoke_epochs if CFG.run_mode == 'smoke' else CFG.full_epochs
print('run mode:', CFG.run_mode, 'epochs:', epochs)

seed_everything(CFG.seed)
baseline_model = BaselinePolypPVT().to(DEVICE)
baseline_optimizer = make_baseline_optimizer(baseline_model)
baseline_history = train_model('polyp_pvt', baseline_model, train_loader_base, val_loader, baseline_optimizer, epochs)

torch.cuda.empty_cache() if torch.cuda.is_available() else None

seed_everything(CFG.seed)
cfp_model = CFPPolypPVT(cfp_layer).to(DEVICE)
cfp_optimizer = make_cfp_optimizer(cfp_model)
cfp_history = train_model('cfp_polyp_pvt', cfp_model, train_loader_cfp, val_loader, cfp_optimizer, epochs)

cfp_model.cfp.export_params(str(OUT_DIR / 'cfp_params_after_training.pt'))


run mode: smoke epochs: 1


/Users/wonderalexandre/GitHub/mtlearn/external_models/Polyp-PVT/lib/pvt.py:91: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  edge = F.upsample(edge, (x.size()[-2], x.size()[-1]))


{'epoch': 1, 'model': 'polyp_pvt', 'train_loss': 4.230205774307251, 'val_soft_dice': 0.10606937110424042, 'val_soft_iou': 0.056411441415548325, 'val_dice': 1.2608240840261686e-11, 'val_iou': 1.2608240840261686e-11, 'val_precision': 1.0, 'val_recall': 1.2608240840261686e-11, 'val_mae': 0.38553231954574585, 'seconds': 11.563663244247437}


{'epoch': 1, 'model': 'cfp_polyp_pvt', 'train_loss': 4.2713282108306885, 'val_soft_dice': 0.1064540445804596, 'val_soft_iou': 0.056633394211530685, 'val_dice': 1.2608240840261686e-11, 'val_iou': 1.2608240840261686e-11, 'val_precision': 1.0, 'val_recall': 1.2608240840261686e-11, 'val_mae': 0.3888355493545532, 'seconds': 1.7698071002960205}


## Final Test Metrics and Comparison

In [15]:
@torch.no_grad()
def load_best_and_evaluate(model: nn.Module, checkpoint_path: Path, loader: DataLoader) -> dict[str, Any]:
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint['model'])
    metrics = evaluate(model, loader)
    return {'checkpoint_epoch': checkpoint.get('epoch'), **metrics}


baseline_test = load_best_and_evaluate(baseline_model, OUT_DIR / 'polyp_pvt_best.pt', test_loader)
cfp_test = load_best_and_evaluate(cfp_model, OUT_DIR / 'cfp_polyp_pvt_best.pt', test_loader)
comparison = [
    {'model': 'polyp_pvt', **baseline_test},
    {'model': 'cfp_polyp_pvt', **cfp_test},
]
print(json.dumps(comparison, indent=2))
with (OUT_DIR / 'test_comparison.json').open('w') as f:
    json.dump(comparison, f, indent=2)
with (OUT_DIR / 'test_comparison.csv').open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(comparison[0].keys()))
    writer.writeheader()
    writer.writerows(comparison)


[
  {
    "model": "polyp_pvt",
    "checkpoint_epoch": 1,
    "soft_dice": 0.1792222112417221,
    "soft_iou": 0.10486090928316116,
    "dice": 1.6543542577518444e-11,
    "iou": 1.6543542577518444e-11,
    "precision": 1.0,
    "recall": 1.6543542577518444e-11,
    "mae": 0.4310532808303833
  },
  {
    "model": "cfp_polyp_pvt",
    "checkpoint_epoch": 1,
    "soft_dice": 0.1816827654838562,
    "soft_iou": 0.10660188645124435,
    "dice": 1.6543542577518444e-11,
    "iou": 1.6543542577518444e-11,
    "precision": 1.0,
    "recall": 1.6543542577518444e-11,
    "mae": 0.43292197585105896
  }
]


## Qualitative Predictions

This cell saves a small qualitative grid per model for inspection.

In [16]:
import matplotlib.pyplot as plt


@torch.no_grad()
def save_qualitative_grid(model: nn.Module, loader: DataLoader, name: str, max_items: int = 4) -> None:
    model.eval()
    rows = []
    for batch in loader:
        model_input, mask, names = unpack_batch(batch)
        images = model_input[0] if isinstance(model_input, (tuple, list)) else model_input
        outputs = model(model_input)
        prob = torch.sigmoid(combined_logits(outputs, mask.shape[-2:])).cpu()
        for i in range(images.shape[0]):
            rows.append((images[i].cpu(), mask[i].cpu(), prob[i].cpu(), names[i] if names is not None else str(len(rows))))
            if len(rows) >= max_items:
                break
        if len(rows) >= max_items:
            break

    fig, axes = plt.subplots(len(rows), 3, figsize=(9, 3 * len(rows)))
    if len(rows) == 1:
        axes = np.expand_dims(axes, axis=0)
    for r, (image, mask, prob, sample_name) in enumerate(rows):
        axes[r, 0].imshow(image.permute(1, 2, 0).numpy())
        axes[r, 0].set_title(sample_name)
        axes[r, 1].imshow(mask.squeeze().numpy(), cmap='gray', vmin=0, vmax=1)
        axes[r, 1].set_title('GT')
        axes[r, 2].imshow(prob.squeeze().numpy(), cmap='magma', vmin=0, vmax=1)
        axes[r, 2].set_title(name)
        for c in range(3):
            axes[r, c].axis('off')
    fig.tight_layout()
    path = OUT_DIR / f'{name}_qualitative.png'
    fig.savefig(path, dpi=160)
    plt.close(fig)
    print('saved:', path)


save_qualitative_grid(baseline_model, test_loader, 'polyp_pvt')
save_qualitative_grid(cfp_model, test_loader, 'cfp_polyp_pvt')


/Users/wonderalexandre/GitHub/mtlearn/external_models/Polyp-PVT/lib/pvt.py:91: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  edge = F.upsample(edge, (x.size()[-2], x.size()[-1]))


saved: /Users/wonderalexandre/GitHub/mtlearn/notebooks/experiments/out_kvasir_polyp_pvt_cfp/polyp_pvt_qualitative.png


saved: /Users/wonderalexandre/GitHub/mtlearn/notebooks/experiments/out_kvasir_polyp_pvt_cfp/cfp_polyp_pvt_qualitative.png


## Notes for Full Runs

For final results, set `ExperimentConfig.run_mode` to `'full'` and rerun from the top. On MPS, keep `384x384` unless you patch Polyp-PVT's SAM adaptive pooling to run on CPU; the original `352x352` size fails in PyTorch MPS. Recommended next controls:

- repeat seeds `42`, `1337`, `2026`;
- compare `TREE_OF_SHAPES + SHAPE` against `MAX_TREE + AREA/COMPACTNESS/GRAY_HEIGHT`;
- add an identity-frozen CFP sanity baseline;
- add offline deterministic augmentations only after the no-augmentation cache path is validated.